In [1]:
import pandas as pd 
import numpy as np 

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
df = pd.read_csv('titanic.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], inplace=True)
# inplace=True    #  Modifies the original DataFrame, No new DataFrame is created
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


<h3 style="color:brown">Train Test Split</h3>

In [4]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df['Survived'], test_size=0.3, random_state=0) 

X_train.shape, X_test.shape

((623, 7), (268, 7))

In [5]:
df.isnull().sum()    # to check for missing values

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

<h3 style="color:brown">Imputer</h3>

In [6]:
# Appling Imputer 
si_age = SimpleImputer()    # default strategy='mean' 
si_embarked = SimpleImputer(strategy='most_frequent')    # replaces missing values by most frequent ones 

X_train_age = si_age.fit_transform(X_train[['Age']])
X_test_age = si_age.transform(X_test[['Age']])

X_train_embarked = si_embarked.fit_transform(X_train[['Embarked']])
X_test_embarked = si_embarked.transform(X_test[['Embarked']])          # all are numpy arrays

In [7]:
np.unique(X_train_embarked)

array(['C', 'Q', 'S'], dtype=object)

<h3 style="color:brown">One Hot Encoding</h3>

In [8]:
# OneHotEncoder on Sex and Embarked 
ohe_sex = OneHotEncoder(sparse_output=False,dtype=int, handle_unknown='ignore')   
ohe_embarked = OneHotEncoder(sparse_output=False, dtype=int, handle_unknown='ignore') 
# handle_ignore='ignore'    #  if in test, you encounter any new category, it will ignore it 
# we didn't used drop='first' because we gonna use decision tree, and on that multicollinearity doesn't have any impact on that. 

X_train_sex = ohe_sex.fit_transform(X_train[['Sex']])
X_test_sex = ohe_sex.transform(X_test[['Sex']])

# X_train['Embarked'] still have missing value, 
# we have applied simple imputer but the updated Embarked data is stored in numpy array 'X_train_Embarked'  
# so we fit and transform using X_train_embarked
X_train_embarked = ohe_embarked.fit_transform(X_train_embarked)         
X_test_embarked = ohe_embarked.transform(X_test_embarked)

In [9]:
ohe_embarked.categories_

[array(['C', 'Q', 'S'], dtype=object)]

In [10]:
X_train_embarked

array([[0, 0, 1],
       [1, 0, 0],
       [0, 0, 1],
       ...,
       [0, 1, 0],
       [0, 0, 1],
       [0, 0, 1]])

In [11]:
X_train_rem = X_train.drop(columns=['Sex','Age','Embarked'], axis=1)    # creating a numpy array of remaining columns
X_test_rem = X_test.drop(columns=['Sex','Age','Embarked'], axis=1)    # creating a numpy array of remaining columns

In [12]:
X_train_rem

,Pclass,SibSp,Parch,Fare
857,1,0,0,26.5500
52,1,1,0,76.7292
386,3,5,2,46.9000
124,1,0,1,77.2875
578,3,1,0,14.4583
...,...,...,...,...
835,1,1,1,83.1583
192,3,1,0,7.8542
629,3,0,0,7.7333
559,3,1,0,17.4000


In [13]:
X_train_tnf = np.concat((X_train_age, X_train_sex, X_train_rem, X_train_embarked), axis=1)
X_test_tnf = np.concat((X_test_age, X_test_sex, X_test_rem, X_test_embarked), axis=1)

In [14]:
X_train_tnf.shape

(623, 10)

<h3 style="color:brown">Decision Tree</h3>

In [15]:
dt = DecisionTreeClassifier() 

# train the model using training data
dt.fit(X_train_tnf, y_train)

# predict the output for test data 
y_pred = dt.predict(X_test_tnf)

In [16]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, y_pred)

0.7910447761194029

<h3 style="color:brown">Pickle Library</h3>

In [18]:
import pickle

# pickle is a Python library used to save and load Python objects
# It converts objects into a file format that can be stored on disk

# Think of pickle like:
# “Save this trained object exactly as it is, so I can use it later”

In [23]:
#  as we will give the input, no need to export simple imputer as we won't miss any value in input to give.

#  exporting objects to the given location 'models/...'
pickle.dump(ohe_sex, open('models/ohe_sex.pkl','wb'))                    # ohe_sex.pkl : Encode Sex column correctly
pickle.dump(ohe_embarked, open('models/ohe_embarked.pkl', 'wb'))         # ohe_embarked.pkl : Encode Embarked column correctly
pickle.dump(dt , open('models/decision_tree.pkl', 'wb'))                  # decision_tree.pkl : Make predictions

# ohe_sex → a trained OneHotEncoder for the Sex column
# open('models/ohe_sex.pkl', 'wb')
    # 'models/' → folder where you store models
    # 'ohe_sex.pkl' → file name
    # 'wb' → write binary mode (required for pickle)
# pickle.dump(object, file):  Saves the object into the file  
# During prediction, new data must be encoded in the same way
# If you train a new encoder again → column order may change
# So we reuse the same trained encoder